# DINOv3 ViT-B/16 wine-label retrieval

End-to-end pipeline after YOLO label detection: build a leakage-safe identity split, transform label crops to valid DINOv3 inputs, train a retrieval embedding, validate against one reference bottle per wine, export the gallery, and benchmark inference.

The default strategy first trains only a small retrieval head and then fine-tunes the final two transformer blocks. Horizontal and vertical flips are deliberately disabled because mirrored label text is not physically realistic.

## 1. Environment

Use the project kernel `Python (VIscaNEr)`. The local environment is already installed. On a fresh machine or Kaggle, run the next cell once. PyTorch is not force-reinstalled on macOS.

In [ ]:
# Fresh environment only; uncomment when needed.
# %pip install -r requirements.txt

In [ ]:
from pathlib import Path
import json
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd
import torch
from PIL import Image

LOCAL_PROJECT_ROOT = Path("/Users/konstantinmelnikov/Desktop/work/VIscaNEr")
RUNNING_ON_KAGGLE = Path("/kaggle/input").exists()
PROJECT_ROOT = Path("/kaggle/working/VIscaNEr") if RUNNING_ON_KAGGLE else LOCAL_PROJECT_ROOT
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dinov3_retrieval import (
    DINOv3RetrievalModel, PipelineConfig, WineRetrievalDataset,
    benchmark_model, build_transforms, choose_device, create_eval_loader,
    environment_summary, evaluate_splits, load_local_dinov3_backbone,
    load_trained_model, prepare_retrieval_index, train_pipeline,
    visualize_transforms,
)

DEVICE = choose_device("auto")
environment_summary(DEVICE)

## 2. Paths and training configuration

Locally, all paths resolve inside the project. On Kaggle, attach the two private datasets described in `KAGGLE_DINOV3.md`; only the input path overrides below change. Outputs always go to `/kaggle/working/VIscaNEr`.

In [ ]:
if RUNNING_ON_KAGGLE:
    KAGGLE_DATA = Path("/kaggle/input/viscaner-dinov3-data")
    KAGGLE_WEIGHTS = Path("/kaggle/input/viscaner-dinov3-vitb16-weights/model.safetensors")
    CONFIG = PipelineConfig(
        project_root=str(PROJECT_ROOT),
        weights_path=str(KAGGLE_WEIGHTS),
        crops_metadata_path=str(KAGGLE_DATA / "crops_metadata.csv"),
        bottle_manifest_path=str(KAGGLE_DATA / "bottle_images_manifest.csv"),
        crops_root=str(KAGGLE_DATA / "crops"),
        refs_root=str(KAGGLE_DATA / "refs"),
        num_workers=4,
        eval_batch_size=64,
        device="cuda",
    ).resolved()
else:
    CONFIG = PipelineConfig(project_root=str(PROJECT_ROOT)).resolved()

CONFIG

### Recommended defaults

- input: 224×224 RGB, which is divisible by the 16-pixel patch size; aspect ratio is preserved with square padding;
- two independent but mild views per training image;
- Stage 1: frozen 86M-parameter backbone, 5 epochs;
- Stage 2: unfreeze the last 2 blocks, 12 epochs at `1e-5`;
- objective: supervised contrastive loss plus normalized identity classification;
- validation: crop query → clean bottle reference, reported separately for seen identities, unseen identities, and low-confidence YOLO crops.

## 3. Build and validate the retrieval index

The split is identity-aware. No held-out `val_unseen` wine may appear in training. Two successful images per seen wine are reserved as `val_seen`; low-confidence YOLO crops form a separate hard benchmark. Failed detections are excluded because they contain no trustworthy label crop.

In [ ]:
index_df, split_summary = prepare_retrieval_index(CONFIG, validate_files=True)
display(pd.DataFrame({
    "rows": split_summary["split_rows"],
    "identities": split_summary["split_identities"],
}).fillna(0).astype(int))
assert split_summary["leakage_check_train_vs_unseen"] == 0
split_summary

In [ ]:
# Dimension sanity check on a reproducible sample. Every image is converted to RGB,
# padded to a square, resized to 224x224, converted to tensor, and normalized.
sample = index_df[index_df.split.eq("train")].sample(500, random_state=CONFIG.seed)
widths, heights = [], []
for row in sample.itertuples():
    path = Path(CONFIG.refs_root) / row.filename if row.path_kind == "reference" else Path(CONFIG.crops_root) / row.status / row.filename
    with Image.open(path) as image:
        widths.append(image.width)
        heights.append(image.height)
dimension_summary = pd.DataFrame({"width": widths, "height": heights})
display(dimension_summary.describe().round(1))
dimension_summary.plot(kind="hist", bins=30, alpha=0.65, figsize=(9, 4), title="Raw crop dimensions");

## 4. Inspect augmentations

Check that text remains readable and the label is not cropped away. There are no flips. Geometry, illumination, colour, perspective and blur are intentionally mild.

In [ ]:
visualize_transforms(CONFIG, n=6)

## 5. Strict checkpoint and tensor smoke test

This verifies the local safetensors checkpoint against the exact ViT-B/16 architecture: 12 blocks, 768 hidden units, 12 heads, 4 register tokens and 16×16 patches. No network download is needed.

In [ ]:
backbone = load_local_dinov3_backbone(CONFIG.weights_path, CONFIG.image_size)
model = DINOv3RetrievalModel(
    backbone,
    num_classes=int(index_df.label_id.max()) + 1,
    embedding_dim=CONFIG.embedding_dim,
    projection_hidden_dim=CONFIG.projection_hidden_dim,
    ce_temperature=CONFIG.ce_temperature,
).to(DEVICE)
model.set_backbone_trainable(0)
_, eval_transform = build_transforms(CONFIG.image_size)
smoke_records = index_df[index_df.split.eq("train")].head(2)
smoke_dataset = WineRetrievalDataset(smoke_records, CONFIG.crops_root, CONFIG.refs_root, eval_transform)
smoke_batch = torch.stack([smoke_dataset[i]["image"] for i in range(len(smoke_dataset))]).to(DEVICE)
with torch.inference_mode():
    embeddings, logits = model(smoke_batch)
print(f"Backbone parameters: {sum(p.numel() for p in backbone.parameters()):,}")
print("Input:", tuple(smoke_batch.shape), "Embedding:", tuple(embeddings.shape), "Logits:", tuple(logits.shape))
del model, backbone, smoke_batch
if DEVICE.type == "mps":
    torch.mps.empty_cache()
elif DEVICE.type == "cuda":
    torch.cuda.empty_cache()

## 6. Train

Set `RUN_TRAINING=True` when ready. The best checkpoint is selected by seen-identity Recall@1, while unseen identities remain a separate generalisation test. Checkpoints are written after every epoch. On Kaggle, use the CLI command from the guide for a run that continues independently of this notebook page.

In [ ]:
RUN_TRAINING = False  # Change to True for the full 5 + 12 epoch run.
if RUN_TRAINING:
    train_result = train_pipeline(CONFIG)
    display(train_result)
else:
    print("Training skipped. Set RUN_TRAINING=True or use train_dinov3_retrieval.py all.")

## 7. Final validation and gallery export

Metrics: Recall@1/5/10, mean reciprocal rank, median rank and mean rank. The exported `gallery_embeddings.pt` contains one normalized embedding per wine and is the artefact used by the production nearest-neighbour search.

In [ ]:
BEST_CHECKPOINT = Path(CONFIG.models_dir) / "best.pt"
if BEST_CHECKPOINT.exists():
    trained_model, checkpoint = load_trained_model(BEST_CHECKPOINT, CONFIG.weights_path, DEVICE)
    final_metrics, retrieval_details = evaluate_splits(trained_model, index_df, CONFIG, DEVICE)
    display(pd.DataFrame(final_metrics).T)
else:
    print(f"No checkpoint yet: {BEST_CHECKPOINT}. Run training first.")

## 8. Inference benchmark

The benchmark performs warm-up iterations and synchronises CUDA or MPS before timing. Reported latency is model-only per image for the configured batch.

In [ ]:
if BEST_CHECKPOINT.exists():
    gallery_records = index_df[index_df.split.eq("gallery")].head(256)
    benchmark_loader = create_eval_loader(gallery_records, CONFIG)
    benchmark = benchmark_model(trained_model, benchmark_loader, DEVICE, warmup=10, iterations=100)
    display(pd.Series(benchmark, name="value").to_frame())
else:
    print("Benchmark skipped until best.pt exists.")

## 9. What to inspect after training

- `models/dinov3_retrieval/best.pt` — deployable retrieval model;
- `runs/dinov3_retrieval/dinov3_vitb16_wine_retrieval/final_metrics.json` — seen, unseen and hard-query metrics;
- `gallery_embeddings.pt` — reference index for production search;
- `retrieval_failures.png` — worst query/reference mistakes;
- `training_curves.png` and `history.csv` — optimisation diagnostics.

If seen Recall@1 is high but unseen Recall@1 is weak, reduce fine-tuning or add more identity diversity. If both are weak, inspect YOLO crops and hard negatives. If low-confidence performance is weak, improve the detector before adding aggressive DINO augmentation.

In [ ]:
run_dir = Path(CONFIG.runs_dir) / CONFIG.run_name
summary = {
    "best_model": str(Path(CONFIG.models_dir) / "best.pt"),
    "last_model": str(Path(CONFIG.models_dir) / "last.pt"),
    "retrieval_index": CONFIG.index_path,
    "gallery_embeddings": str(run_dir / "gallery_embeddings.pt"),
    "metrics": str(run_dir / "final_metrics.json"),
    "split_summary": split_summary,
}
summary